# LeetCode 2239 — Find Closest Number to Zero bằng Python

Link đề: https://leetcode.com/problems/find-closest-number-to-zero/ · Độ khó: **Easy** · Chủ đề: Array, Sorting

Skeleton LeetCode:

```python
class Solution:
    def findClosestNumber(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu "gần `0` nhất" nghĩa là **giá trị tuyệt đối nhỏ nhất**, và luật **hòa** (tie-break): chọn số **lớn hơn**.
2. Dùng `abs`, `min` với tham số `key`, `lambda`, và so sánh **tuple** để gộp nhiều tiêu chí.
3. Viết 5 cách: brute force `O(n²)`, một lượt cập nhật tốt nhất, `min` với khóa `(abs(x), -x)`, mẹo `min abs` + tra `set`, và tách dương/âm.
4. Tự trace, tự test (vét cạn trên miền nhỏ + random) và nhận diện lỗi hòa kinh điển.
5. Nắm pattern "chọn phần tử tốt nhất theo tiêu chí" cho cả họ bài mảng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có độ dài `n`. Hãy trả về số trong `nums` **gần `0` nhất**. Nếu có nhiều số cùng khoảng cách tới `0` (ví dụ `-2` và `2`), trả về số **lớn hơn**.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[-4,-2,1,4,8]` | `1` | Khoảng cách tới `0`: `4,2,1,4,8` ⇒ nhỏ nhất là `1`. |
| 2 | `[2,-1,1]` | `1` | `-1` và `1` đều cách `0` một đơn vị; chọn số lớn hơn là `1`. |

Constraints:

- `1 <= n <= 1000`
- `-10^5 <= nums[i] <= 10^5`

In [1]:
examples = [([-4, -2, 1, 4, 8], 1), ([2, -1, 1], 1)]
for nums, expected in examples:
    print("nums =", nums, "| khoảng cách tới 0:", [abs(x) for x in nums], "| mong đợi =", expected)

nums = [-4, -2, 1, 4, 8] | khoảng cách tới 0: [4, 2, 1, 4, 8] | mong đợi = 1
nums = [2, -1, 1] | khoảng cách tới 0: [2, 1, 1] | mong đợi = 1


## 3. Kiến thức nền

### 3.1. Giá trị tuyệt đối `abs`

`abs(x)` là khoảng cách từ `x` đến `0`: `abs(-4) = 4`, `abs(4) = 4`, `abs(0) = 0`. Hai số đối nhau như `-2` và `2` có cùng `abs`.

In [2]:
print(abs(-4), abs(4), abs(0))
print([abs(x) for x in [-4, -2, 1, 4, 8]])

4 4 0
[4, 2, 1, 4, 8]


### 3.2. `min` / `max` với tham số `key`

`min(iterable, key=f)` trả phần tử có `f(phần tử)` nhỏ nhất (không phải phần tử nhỏ nhất). Ví dụ: `min(nums, key=abs)` trả số có `abs` nhỏ nhất. Khi có nhiều phần tử hòa, `min` trả phần tử **đứng đầu tiên** trong dãy.

In [3]:
nums = [-4, -2, 1, 4, 8]
print(min(nums, key=abs))
print(min([2, -1, 1], key=abs))      # hòa giữa -1 và 1 -> trả -1 (đứng trước 1)
print(min(["bb", "a", "ccc"], key=len))

1
-1
a


### 3.3. `lambda` — hàm nhỏ viết tại chỗ

`lambda x: biểu_thức` là hàm vô danh. `min(nums, key=lambda x: abs(x))` tương đương `key=abs`, nhưng với `lambda` ta có thể viết biểu thức phức tạp hơn.

In [4]:
square = lambda x: x * x
print(square(5))
print(min([3, -5, 2], key=lambda x: x * x))

25
2


### 3.4. So sánh tuple: nhiều tiêu chí cùng lúc

Python so sánh tuple **từng phần tử từ trái sang phải**: phần tử đầu quyết định, chỉ khi bằng nhau mới xét phần tử kế tiếp. Vậy `(abs(x), -x)` nghĩa là: ưu tiên khoảng cách nhỏ; nếu hòa thì ưu tiên `-x` nhỏ, tức `x` **lớn**.

In [5]:
print((1, 5) < (2, 0))        # True: phần tử đầu quyết định
print((1, 5) < (1, 7))        # True: hòa phần tử đầu, xét tiếp
for x in [-1, 1]:
    print(x, "->", (abs(x), -x))
print(min([-1, 1], key=lambda x: (abs(x), -x)))

True
True
-1 -> (1, 1)
1 -> (1, -1)
1


### 3.5. `set` và toán tử `in`

`s = set(nums)` cho tra cứu `x in s` trung bình `O(1)`. Ta sẽ dùng để kiểm tra "số dương có mặt trong mảng không".

In [6]:
s = set([-4, -2, 1, 4, 8])
print(1 in s, -1 in s, 0 in s)

True False False


## 4. Ý tưởng nền tảng — tính bằng tay

Ví dụ `nums = [2, -1, 1]`:

| Số | Khoảng cách `abs` |
|---|---|
| 2 | 2 |
| -1 | 1 |
| 1 | 1 |

Khoảng cách nhỏ nhất là `1`, có hai ứng viên: `-1` và `1`. Luật hòa chọn số **lớn hơn** ⇒ `1`.

Ví dụ `nums = [-4,-2,1,4,8]`: khoảng cách `4,2,1,4,8` → nhỏ nhất `1` chỉ có số `1` ⇒ đáp án `1`.

**Quy trình tay:** (1) tìm khoảng cách nhỏ nhất `d`; (2) trong các số có khoảng cách `d`, lấy số lớn nhất. Nhưng số có khoảng cách `d` chỉ có thể là `d` hoặc `-d`; do đó **nếu `d` có trong mảng thì chọn `d`, ngược lại chọn `-d`**.

In [7]:
nums = [2, -1, 1]
d = min(abs(x) for x in nums)
candidates = [x for x in nums if abs(x) == d]
print("d =", d, "| ứng viên:", candidates, "| đáp án:", max(candidates))

d = 1 | ứng viên: [-1, 1] | đáp án: 1


## 5. Cách 1 — Brute Force: kiểm tra từng phần tử với mọi phần tử khác

Phần tử `x` là đáp án nếu **không có `y` nào "tốt hơn"**, tức không tồn tại `y` với `abs(y) < abs(x)`, hoặc `abs(y) == abs(x)` mà `y > x`. Duyệt từng `x`, kiểm tra với mọi `y`.

In [8]:
class SolutionBrute:
    def findClosestNumber(self, nums):
        for x in nums:
            best = True
            for y in nums:
                if abs(y) < abs(x) or (abs(y) == abs(x) and y > x):
                    best = False          # y tốt hơn x
                    break
            if best:
                return x

sol = SolutionBrute()
print(sol.findClosestNumber([-4, -2, 1, 4, 8]))
print(sol.findClosestNumber([2, -1, 1]))

1
1


**Độ phức tạp:** hai vòng lồng ⇒ Time `O(n²)`, Space `O(1)`. Với `n = 1000` là `10^6` phép so sánh, vẫn nhanh; nhưng không tối ưu.

**Nút thắt:** mỗi `x` được so với mọi `y`, trong khi chỉ cần **nhớ ứng viên tốt nhất hiện tại** và so với nó (một lượt).

## 6. Ý tưởng thuật toán — bước đột phá

**Insight:** "tốt hơn" là một quan hệ thứ tự **toàn phần** theo cặp khóa `(abs(x), -x)`: nhỏ hơn thì tốt hơn. Do đó đáp án chính là **phần tử nhỏ nhất theo khóa đó** — bài toán chọn phần tử cực trị, chỉ cần một lượt.

Quy tắc "nếu… thì…" khi duyệt với ứng viên `best`:

- **Nếu** `abs(x) < abs(best)` **thì** `best = x` (gần hơn).
- **Nếu** `abs(x) == abs(best)` **và** `x > best` **thì** `best = x` (hòa, chọn lớn hơn).
- **Ngược lại** giữ nguyên.

**Invariant:** sau khi xử lý `nums[0..i]`, `best` là số tốt nhất trong các số đã duyệt.

## 7. Các cách tiếp theo

### Cách 2 — Một lượt, cập nhật `best`

Chính là quy tắc ở mục 6. Bắt đầu `best = nums[0]`.

In [9]:
class SolutionOnePass:
    def findClosestNumber(self, nums):
        best = nums[0]
        for x in nums[1:]:
            if abs(x) < abs(best) or (abs(x) == abs(best) and x > best):
                best = x
        return best

sol = SolutionOnePass()
print(sol.findClosestNumber([-4, -2, 1, 4, 8]))
print(sol.findClosestNumber([2, -1, 1]))

1
1


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 3 — `min` với khóa `(abs(x), -x)`

Dùng ngay ý "tốt nhất theo khóa tuple". Rất ngắn và đọc được như văn: "lấy số có (khoảng cách, -giá trị) nhỏ nhất".

In [10]:
class SolutionMinKey:
    def findClosestNumber(self, nums):
        return min(nums, key=lambda x: (abs(x), -x))

sol = SolutionMinKey()
print(sol.findClosestNumber([-4, -2, 1, 4, 8]))
print(sol.findClosestNumber([2, -1, 1]))

1
1


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Tìm khoảng cách nhỏ nhất `d`, rồi "`d` hay `-d`?"

Bước 1: `d = min(abs(x))`. Bước 2: số có khoảng cách `d` chỉ có thể là `d` hoặc `-d`. Vì luật hòa ưu tiên số lớn, nếu `d` có mặt thì trả `d`, nếu không thì trả `-d`. (Khi `d = 0` thì `d` có mặt, trả `0`.)

In [11]:
class SolutionAbsSet:
    def findClosestNumber(self, nums):
        d = min(abs(x) for x in nums)
        return d if d in set(nums) else -d

sol = SolutionAbsSet()
print(sol.findClosestNumber([-4, -2, 1, 4, 8]))
print(sol.findClosestNumber([2, -1, 1]))
print(sol.findClosestNumber([-3, 7, -9]))

1
1
-3


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho `set` (có thể thay bằng `d in nums` để `O(1)` bộ nhớ, `O(n)` thời gian thêm).

### Cách 5 — Tách hai phía: âm gần `0` nhất và không âm gần `0` nhất

Số không âm gần `0` nhất là `min` của các `x >= 0`. Số âm gần `0` nhất là `max` của các `x < 0`. So sánh hai ứng viên: nếu `abs(âm) < dương` thì chọn âm, ngược lại chọn dương (hòa chọn dương vì lớn hơn).

In [12]:
class SolutionSplit:
    def findClosestNumber(self, nums):
        pos = [x for x in nums if x >= 0]
        neg = [x for x in nums if x < 0]
        if not neg:
            return min(pos)
        if not pos:
            return max(neg)
        p, q = min(pos), max(neg)
        return q if -q < p else p          # hòa (-q == p) -> chọn p (lớn hơn)

sol = SolutionSplit()
print(sol.findClosestNumber([-4, -2, 1, 4, 8]))
print(sol.findClosestNumber([2, -1, 1]))
print(sol.findClosestNumber([-5, -2, -9]))

1
1
-2


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho hai danh sách phụ. Ưu điểm: mỗi nửa trục là một bài "tìm cực trị" đơn giản, dễ mở rộng thành "gần `target` bất kỳ nhất".

## 8. TRACE từng bước

Trace cách 2 với `nums = [2, -1, 1]`:

| Bước | x | best trước | `abs(x) < abs(best)`? | `abs(x) == abs(best)` và `x > best`? | best sau |
|---|---|---|---|---|---|
| khởi tạo | | | | | `2` |
| 1 | `-1` | `2` | `1 < 2` có | | `-1` |
| 2 | `1` | `-1` | `1 < 1` không | `1 == 1` và `1 > -1` có | `1` |

Kết quả `1`. Hàm dưới in từng bước.

In [13]:
def trace_one_pass(nums):
    best = nums[0]
    print(f"khởi tạo best = {best}")
    for x in nums[1:]:
        if abs(x) < abs(best):
            why = "gần hơn"
            best = x
        elif abs(x) == abs(best) and x > best:
            why = "hòa, x lớn hơn"
            best = x
        else:
            why = "giữ nguyên"
        print(f"x={x:>3} | abs(x)={abs(x)} | {why:<15} | best={best}")
    return best

print(trace_one_pass([2, -1, 1]))

khởi tạo best = 2
x= -1 | abs(x)=1 | gần hơn         | best=-1
x=  1 | abs(x)=1 | hòa, x lớn hơn  | best=1
1


In [14]:
print(trace_one_pass([-4, -2, 1, 4, 8]))

khởi tạo best = -4
x= -2 | abs(x)=2 | gần hơn         | best=-2
x=  1 | abs(x)=1 | gần hơn         | best=1
x=  4 | abs(x)=4 | giữ nguyên      | best=1
x=  8 | abs(x)=8 | giữ nguyên      | best=1
1


## 9. Code nộp LeetCode

Dùng cách 2 (một lượt): rõ ràng, `O(n)`, `O(1)` bộ nhớ.

In [15]:
from typing import List

class Solution:
    def findClosestNumber(self, nums: List[int]) -> int:
        best = nums[0]
        for x in nums[1:]:
            if abs(x) < abs(best) or (abs(x) == abs(best) and x > best):
                best = x
        return best

print(Solution().findClosestNumber([-4, -2, 1, 4, 8]))
print(Solution().findClosestNumber([2, -1, 1]))

1
1


## 10. Vì sao thuật toán đúng?

- **Thứ tự toàn phần:** so sánh theo `(abs(x), -x)` là một thứ tự toàn phần trên các số nguyên (hai số khác nhau luôn có khóa khác nhau, vì `abs` giống nhau thì `-x` khác nhau). Vậy phần tử nhỏ nhất theo khóa là **duy nhất**.
- **Quy tắc cập nhật khớp với thứ tự đó:** `abs(x) < abs(best)` ⇔ khóa của `x` nhỏ hơn ở thành phần đầu; `abs(x) == abs(best)` và `x > best` ⇔ `-x < -best` ở thành phần thứ hai. Ngược lại thì khóa của `x` lớn hơn hoặc bằng, nên không cập nhật.
- **Invariant:** `best` luôn là phần tử nhỏ nhất theo khóa trong tiền tố đã duyệt; thêm `x` mới chỉ có thể thay `best` khi `x` nhỏ hơn theo khóa, nên invariant được giữ. Hết vòng lặp, `best` là cực tiểu của cả mảng.
- **Cách 4:** hai số có cùng khoảng cách `d` chỉ có thể là `d` và `-d`; số lớn hơn là `d` (vì `d >= 0`), nên nếu `d` có mặt chọn `d`, nếu không thì chỉ còn `-d`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` |
| 2. Một lượt | `O(n)` | `O(1)` |
| 3. `min` với `key` | `O(n)` | `O(1)` |
| 4. `min abs` + `set` | `O(n)` | `O(n)` |
| 5. Tách dương/âm | `O(n)` | `O(n)` |

Mọi cách tối ưu đều tuyến tính; sắp xếp `O(n log n)` cũng được nhưng thừa.

## 12. Bộ test

Test ví dụ đề, edge case, vét cạn trên miền nhỏ rồi random lớn đối chiếu mọi cách.

In [16]:
solutions = [SolutionBrute(), SolutionOnePass(), SolutionMinKey(), SolutionAbsSet(), SolutionSplit(), Solution()]

for s in solutions:
    assert s.findClosestNumber([-4, -2, 1, 4, 8]) == 1
    assert s.findClosestNumber([2, -1, 1]) == 1
print("ví dụ đề: OK")

ví dụ đề: OK


In [17]:
for s in solutions:
    assert s.findClosestNumber([5]) == 5                  # một phần tử dương
    assert s.findClosestNumber([-5]) == -5                # một phần tử âm
    assert s.findClosestNumber([0]) == 0
    assert s.findClosestNumber([-3, 3]) == 3              # hòa -> lớn hơn
    assert s.findClosestNumber([3, -3]) == 3              # thứ tự đảo vẫn đúng
    assert s.findClosestNumber([-1, 0, 1]) == 0           # có 0 thì luôn là 0
    assert s.findClosestNumber([-7, -2, -9]) == -2        # toàn âm
    assert s.findClosestNumber([10**5, -10**5]) == 10**5  # biên constraints, hòa
    assert s.findClosestNumber([4, -4, 4, -4, -4]) == 4   # nhiều bản sao
print("edge case: OK")

edge case: OK


In [18]:
from itertools import product

# vét cạn: mọi mảng độ dài 1..4 với giá trị -3..3
total = 0
for n in range(1, 5):
    for nums in product(range(-3, 4), repeat=n):
        nums = list(nums)
        ref = min(nums, key=lambda x: (abs(x), -x))
        for s in solutions:
            assert s.findClosestNumber(nums) == ref, (type(s).__name__, nums)
        total += 1
print("vét cạn", total, "mảng nhỏ: tất cả khớp")

vét cạn 2800 mảng nhỏ: tất cả khớp


In [19]:
import random
random.seed(2239)
for _ in range(1500):
    n = random.randint(1, 30)
    nums = [random.randint(-50, 50) for _ in range(n)]
    results = [s.findClosestNumber(nums) for s in solutions]
    assert len(set(results)) == 1, (nums, results)
print("random đối chiếu 1500 mảng x", len(solutions), "cách: OK")

random đối chiếu 1500 mảng x 6 cách: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên luật hòa (trả `-1` thay vì `1`)

`min(nums, key=abs)` khi hòa trả phần tử **đứng trước**; với `[2, -1, 1]` ra `-1`, trong khi đề đòi số lớn hơn là `1`.

In [20]:
def wrong_tie(nums):
    return min(nums, key=abs)

print("sai :", wrong_tie([2, -1, 1]))
print("đúng:", Solution().findClosestNumber([2, -1, 1]))

sai : -1
đúng: 1


### Lỗi 2 — Trả về khoảng cách thay vì phần tử

`min(abs(x) for x in nums)` cho khoảng cách, không phải số gốc. Với `[-4,-2,1,4,8]` ra `1` đúng "tình cờ", nhưng với `[-2, 5]` trả `2` thay vì `-2`.

In [21]:
def wrong_return_abs(nums):
    return min(abs(x) for x in nums)

print("sai :", wrong_return_abs([-2, 5]))     # 2
print("đúng:", Solution().findClosestNumber([-2, 5]))   # -2

sai : 2
đúng: -2


### Lỗi 3 — So sánh thẳng giá trị thay vì `abs`

`min(nums)` chọn số nhỏ nhất (rất âm), không phải số gần `0`.

In [22]:
def wrong_plain_min(nums):
    return min(nums)

print("sai :", wrong_plain_min([-4, -2, 1, 4, 8]))   # -4
print("đúng:", Solution().findClosestNumber([-4, -2, 1, 4, 8]))

sai : -4
đúng: 1


### Lỗi 4 — Khởi tạo `best = 0`

Gán `best = 0` khi mảng không chứa `0` làm hàm trả về `0` không có trong mảng. Hãy khởi tạo bằng `nums[0]` hoặc dùng "vô cực" cho khoảng cách.

In [23]:
def wrong_init_zero(nums):
    best = 0
    for x in nums:
        if abs(x) < abs(best) or (abs(x) == abs(best) and x > best):
            best = x
    return best

print("sai :", wrong_init_zero([5, 3, 9]))     # 0 (không có trong mảng!)
print("đúng:", Solution().findClosestNumber([5, 3, 9]))

sai : 0
đúng: 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn phần tử **tốt nhất** theo tiêu chí, hòa thì theo tiêu chí phụ" ⇒ `min(..., key=lambda x: (tiêu_chí_chính, tiêu_chí_phụ))`.
- Muốn **lớn hơn** khi hòa thì đặt `-x` làm tiêu chí phụ với `min` (hoặc dùng `max` với khóa đảo dấu).
- "Gần `target` nhất" ⇒ thay `abs(x)` bằng `abs(x - target)`.
- Nhớ: `min(..., key=...)` khi hòa trả phần tử **đầu tiên**; đừng giả định nó chọn số lớn hơn.
- Họ hàng: Find the Closest Palindrome, Closest Binary Search Tree Value (270), Find K Closest Elements (658), Minimum Absolute Difference (1200).

## 15. Biến thể của bài

1. **Gần `target` nhất:** thay `abs(x)` bằng `abs(x - target)`.
2. **Hòa thì chọn số nhỏ hơn:** đổi khóa phụ thành `x`.
3. **Trả về `k` số gần `0` nhất** (LC 658 dạng đơn giản): sắp theo khóa rồi lấy `k` phần tử đầu.
4. **Trả về chỉ số** của phần tử gần `0` nhất.
5. **Tất cả phần tử** đạt khoảng cách nhỏ nhất.

In [24]:
# Biến thể 1 & 3: gần target nhất, và k số gần target nhất
def closest_to_target(nums, target):
    return min(nums, key=lambda x: (abs(x - target), -x))

def k_closest(nums, target, k):
    return sorted(sorted(nums, key=lambda x: (abs(x - target), -x))[:k])

print(closest_to_target([1, 4, 9, 12], 7))      # 9 (hòa 4,9 -> lớn hơn)
print(k_closest([1, 4, 9, 12], 7, 2))

9
[4, 9]


### Mở rộng: sắp xếp + `bisect` (khi phải trả lời **nhiều truy vấn** "gần `target` nhất")

Nếu mảng cố định và có rất nhiều truy vấn, ta sắp xếp **một lần** (`O(n log n)`), mỗi truy vấn dùng tìm kiếm nhị phân `bisect_left` (`O(log n)`) để tìm vị trí chèn `target`; đáp án chỉ có thể là phần tử ngay trước hoặc ngay sau vị trí đó. Với một truy vấn duy nhất (như bài này), cách một lượt `O(n)` vẫn tốt hơn.

In [25]:
from bisect import bisect_left

def closest_sorted(sorted_nums, target):
    i = bisect_left(sorted_nums, target)          # vị trí chèn target
    cands = []
    if i > 0:
        cands.append(sorted_nums[i - 1])
    if i < len(sorted_nums):
        cands.append(sorted_nums[i])
    return min(cands, key=lambda x: (abs(x - target), -x))

arr = sorted([-4, -2, 1, 4, 8])
print(closest_sorted(arr, 0), closest_sorted(arr, 3), closest_sorted(arr, -3))
for t in range(-10, 11):
    assert closest_sorted(arr, t) == closest_to_target(arr, t), t
assert closest_sorted(arr, 0) == Solution().findClosestNumber([-4, -2, 1, 4, 8])
print("bisect khớp cách một lượt trên mọi target trong [-10, 10]")

1 4 -2
bisect khớp cách một lượt trên mọi target trong [-10, 10]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | So mỗi `x` với mọi `y` | `O(n²)` | `O(1)` | Hiểu định nghĩa "tốt nhất" |
| 2. Một lượt | Giữ `best`, cập nhật theo luật | `O(n)` | `O(1)` | **Nộp bài**, nền tảng mọi bài cực trị |
| 3. `min` + `key` tuple | Khóa `(abs(x), -x)` | `O(n)` | `O(1)` | Viết nhanh, rất Pythonic |
| 4. `min abs` + `set` | Trả `d` nếu có, ngược lại `-d` | `O(n)` | `O(n)` | Khi muốn nhận xét toán học |
| 5. Tách dương/âm | Hai ứng viên rồi so sánh | `O(n)` | `O(n)` | Dễ mở rộng sang bài "gần target" |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `Solution().findClosestNumber([-7, 7, 5, -5])` là gì?

**Bài 2 (tính tay).** Tính khoảng cách tới `0` của `[8, -3, 6, 3, -9]` rồi chọn đáp án.

**Bài 3 (điền TODO).** Hoàn thành `key_fn(x)` trả khóa `(abs(x), -x)` dùng cho `min`.

**Bài 4 (tự viết lại).** Viết `closest_to_target(nums, target)` (gần `target` nhất, hòa chọn lớn hơn) không nhìn code mẫu.

**Bài 5 (mini challenge).** Viết `all_closest(nums)` trả về danh sách **tất cả** giá trị phân biệt có khoảng cách nhỏ nhất tới `0`, sắp xếp tăng dần (tối đa 2 phần tử).

In [26]:
# Bài 1: đoán rồi chạy
print(Solution().findClosestNumber([-7, 7, 5, -5]))

5


In [27]:
# Bài 3, 4, 5: TODO - KHÔNG được gọi khi run all
def key_fn(x):
    # TODO: trả về tuple (abs(x), -x)
    pass

def closest_to_target_todo(nums, target):
    # TODO: tìm số gần target nhất; hòa thì lấy số lớn hơn
    pass

def all_closest(nums):
    # TODO: trả về danh sách các giá trị phân biệt có |x| nhỏ nhất, tăng dần
    pass

## 18. Gợi ý 3 mức

**Bài 3**
- Mức 1: Khóa gồm hai thành phần theo thứ tự ưu tiên.
- Mức 2: Thành phần đầu là khoảng cách `abs(x)`.
- Mức 3: `return abs(x), -x`.

**Bài 4**
- Mức 1: Thay `abs(x)` bằng `abs(x - target)`.
- Mức 2: Vẫn dùng `-x` làm khóa phụ để hòa chọn số lớn hơn.
- Mức 3: `return min(nums, key=lambda x: (abs(x - target), -x))`.

**Bài 5**
- Mức 1: Tìm `d = min(abs(x))` trước.
- Mức 2: Lọc các `x` có `abs(x) == d`, bỏ trùng bằng `set`.
- Mức 3: `return sorted({x for x in nums if abs(x) == d})`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [28]:
# Bài 1: hòa giữa -5, 5 (khoảng cách 5) -> 5
assert Solution().findClosestNumber([-7, 7, 5, -5]) == 5

# Bài 2: khoảng cách [8,3,6,3,9] -> nhỏ nhất 3 ở -3 và 3 -> chọn 3
assert [abs(x) for x in [8, -3, 6, 3, -9]] == [8, 3, 6, 3, 9]
assert Solution().findClosestNumber([8, -3, 6, 3, -9]) == 3

# Bài 3
def key_fn_ans(x):
    return abs(x), -x
assert key_fn_ans(-4) == (4, 4)
assert min([-1, 1], key=key_fn_ans) == 1

# Bài 4
def closest_to_target_ans(nums, target):
    return min(nums, key=lambda x: (abs(x - target), -x))
assert closest_to_target_ans([1, 4, 9, 12], 7) == 9
assert closest_to_target_ans([5], 100) == 5
assert closest_to_target_ans([-4, -2, 1, 4, 8], 0) == Solution().findClosestNumber([-4, -2, 1, 4, 8])

# Bài 5
def all_closest_ans(nums):
    d = min(abs(x) for x in nums)
    return sorted({x for x in nums if abs(x) == d})
assert all_closest_ans([2, -1, 1, 1]) == [-1, 1]
assert all_closest_ans([-4, -2, 1, 4, 8]) == [1]
assert all_closest_ans([0, 0, 3]) == [0]
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
from typing import List

class Solution:
    def findClosestNumber(self, nums: List[int]) -> int:
        best = nums[0]
        for x in nums[1:]:
            if abs(x) < abs(best) or (abs(x) == abs(best) and x > best):
                best = x
        return best

# Một dòng tương đương:
#   return min(nums, key=lambda x: (abs(x), -x))
```

### Checklist

- [ ] Khoảng cách tới `0` là `abs(x)`; trả về **phần tử gốc**, không phải khoảng cách.
- [ ] Luật hòa: chọn số **lớn hơn** (`-1` vs `1` → `1`).
- [ ] Khởi tạo `best = nums[0]`, không phải `0`.
- [ ] `min(..., key=abs)` một mình **chưa đủ** vì không xử lý đúng luật hòa.
- [ ] Đã test `[-3,3]`, `[3,-3]`, `[0]`, mảng một phần tử và toàn số âm.
- [ ] Biết dạng tổng quát: gần `target` nhất.